<a href="https://colab.research.google.com/github/Talha-Shahid12/LLM-FOR-AMBULANCE/blob/main/Conversions_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/openai/whisper.git

In [ ]:
!pip install --upgrade --no-deps --force-reinstall git+https://github.com/openai/whisper.git

In [ ]:
!sudo apt update && sudo apt install ffmpeg

In [ ]:
!pip install fastapi nest-asyncio pyngrok uvicorn

In [ ]:
!pip install python-multipart

In [ ]:
!pip install googletrans==4.0.0-rc1

In [ ]:
!pip install gtts

In [ ]:
!pip install playsound


In [ ]:
!pip install pygobject

In [ ]:
!pip install pydub numpy torch

In [22]:
from googletrans import Translator
from gtts import gTTS
import playsound
from google.colab import userdata
from fastapi import FastAPI,Form
import whisper
import nest_asyncio
import uvicorn
import numpy as np
import torch
import os
import io
from pydub import AudioSegment
import nest_asyncio
from pyngrok import ngrok
import uvicorn

In [ ]:
model = whisper.load_model("base")

In [ ]:
#set ngork auth token in colab enviorment varibale
authtoken=userdata.get('authtoken')

In [ ]:
!ngrok authtoken authtoken

In [4]:
class TTS_STT_Service:
    def __init__(self):
        self.translator = Translator()

    def text_to_speech(self, text, output_file="output.mp3"):
        """
        Translates text to Urdu and converts it to speech.
        """
        # Translate to Urdu
        translated_text = self.translator.translate(text, src='en', dest='ur').text
        print(f"Translated Text: {translated_text}")

        # Convert to speech
        tts = gTTS(text=translated_text, lang='ur')
        tts.save(output_file)

        # Play the audio
        playsound.playsound(output_file)
        return output_file

    def speech_to_text(self, file_path, task="translate"):
        """
        Transcribes or translates speech from an audio file.
        """
        result = model.transcribe(file_path, task=task)
        return result["text"]

In [ ]:
class TTS_STT_Controller:
    def __init__(self):
        self.service = TTS_STT_Service()

    async def handle_text_to_voice(self, text: str):
        """
        Handle text-to-voice conversion endpoint.
        """
        output_file = "output.mp3"
        try:
            self.service.text_to_speech(text, output_file)
            return {"message": "Text successfully converted to speech.", "file": output_file}
        except Exception as e:
            return {"error": str(e)}

    async def handle_voice_to_text(self, file: UploadFile):
        """
        Handle voice-to-text transcription endpoint.
        """
        try:
            file_location = f"./{file.filename}"
            with open(file_location, "wb") as buffer:
                buffer.write(await file.read())

            transcription = self.service.speech_to_text(file_location)
            os.remove(file_location)  # Clean up uploaded file
            return {"transcription": transcription}
        except Exception as e:
            return {"error": str(e)}

In [ ]:
controller = TTS_STT_Controller()

In [ ]:
@app.post("/text-to-voice")
async def text_to_voice(text: str = Form(...)):
    """
    API endpoint to convert English text to Urdu speech.
    """
    return await controller.handle_text_to_voice(text)


@app.post("/voice-to-text")
async def voice_to_text(file: UploadFile):
    """
    API endpoint to transcribe voice to text.
    """
    return await controller.handle_voice_to_text(file)

In [ ]:
PORT=8000


In [ ]:
app = FastAPI()

In [ ]:
ngrok_tunnel = ngrok.connect(PORT)
print(f"Public URL: {ngrok_tunnel.public_url}")

nest_asyncio.apply()
uvicorn.run(app, port=PORT)

In [ ]:
#For Testing purpose direct in colab envoirment
!whisper "/content/sample.opus" --model medium --task translate